In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import auc, roc_auc_score

from sklearn.metrics import auc, roc_auc_score, roc_curve

def evaluate_model(y_test, risk_draws, class_labels=None, positive_label=1):
    """
    Evaluate posterior risk draws.

    Binary input:     (draws, patients), risks for positive_label
    Multiclass input: (draws, patients, classes), class order given by class_labels
                      (or inferred from sorted unique y_test labels)
    """
    y = np.asarray(y_test)
    draws = np.asarray(risk_draws, dtype=float)

    if y.ndim != 1:
        raise ValueError("y_test must be one-dimensional.")
    if draws.ndim not in (2, 3) or draws.shape[1] != len(y):
        raise ValueError("risk_draws must have shape (draws, patients) or (draws, patients, classes).")
    if draws.shape[0] == 0 or not np.isfinite(draws).all():
        raise ValueError("risk_draws must contain finite posterior draws.")
    if np.any((draws < 0) | (draws > 1)):
        raise ValueError("Risk probabilities must be between 0 and 1.")

    classes = list(class_labels) if class_labels is not None else list(np.unique(y))
    if len(classes) < 2 or len(set(classes)) != len(classes):
        raise ValueError("Provide at least two unique class labels.")
    if any(not np.any(y == label) for label in classes) or any(label not in classes for label in np.unique(y)):
        raise ValueError("class_labels must match the labels present in y_test.")

    binary_input = draws.ndim == 2
    if binary_input:
        if len(classes) != 2 or positive_label not in classes:
            raise ValueError("Binary draws require two classes and a matching positive_label.")
        positive_index = classes.index(positive_label)
        probabilities = np.empty((draws.shape[0], len(y), 2))
        probabilities[:, :, positive_index] = draws
        probabilities[:, :, 1 - positive_index] = 1 - draws
    else:
        probabilities = draws
        if probabilities.shape[2] != len(classes):
            raise ValueError("The class axis of risk_draws must match class_labels.")
        if not np.allclose(probabilities.sum(axis=2), 1.0, atol=1e-5):
            raise ValueError("Multiclass probabilities must sum to 1 for each patient and draw.")

    y_one_hot = np.column_stack([y == label for label in classes]).astype(int)
    if np.any(y_one_hot.sum(axis=0) == 0) or np.any(y_one_hot.sum(axis=0) == len(y)):
        raise ValueError("Each class must have both positive and negative examples in y_test.")

    def summarize(values):
        return {
            "mean": float(np.mean(values)),
            "median": float(np.median(values)),
            "95%_interval": tuple(np.percentile(values, [2.5, 97.5])),
        }

    # O:E is calculated per class for multiclass outcomes; for binary outcomes,
    # it is reported for the positive/event class.
    oe_classes = [positive_label] if binary_input else classes
    oe_ratios = {}
    observed_counts = {}
    expected_means = {}

    for label in oe_classes:
        class_index = classes.index(label)
        observed = int(np.sum(y == label))
        expected_per_draw = probabilities[:, :, class_index].sum(axis=1)
        if np.any(expected_per_draw <= 0):
            raise ValueError(f"Expected count for class {label!r} is zero in at least one posterior draw.")
        oe_ratios[label] = summarize(observed / expected_per_draw)
        observed_counts[label] = observed
        expected_means[label] = float(expected_per_draw.mean())

    # AUC is calculated for every posterior draw, then summarized.
    auc_draws = {"macro": [], "micro": []}
    for probability_draw in probabilities:
        class_aucs = [
            roc_auc_score(y_one_hot[:, i], probability_draw[:, i])
            for i in range(len(classes))
        ]
        auc_draws["macro"].append(float(np.mean(class_aucs)))
        auc_draws["micro"].append(
            float(roc_auc_score(y_one_hot.ravel(), probability_draw.ravel()))
        )

    # Plot macro and micro ROC curves using mean posterior probabilities.
    mean_probabilities = probabilities.mean(axis=0)
    class_curves = [
        roc_curve(y_one_hot[:, i], mean_probabilities[:, i])
        for i in range(len(classes))
    ]
    all_fpr = np.unique(np.concatenate([curve[0] for curve in class_curves]))
    mean_tpr = np.mean(
        [np.interp(all_fpr, fpr, tpr) for fpr, tpr, _ in class_curves],
        axis=0,
    )
    macro_curve_auc = auc(all_fpr, mean_tpr)

    micro_fpr, micro_tpr, _ = roc_curve(
        y_one_hot.ravel(), mean_probabilities.ravel()
    )
    micro_curve_auc = auc(micro_fpr, micro_tpr)

    plt.figure(figsize=(7, 6))
    plt.plot(all_fpr, mean_tpr, label=f"Macro-average (AUC = {macro_curve_auc:.3f})")
    plt.plot(micro_fpr, micro_tpr, label=f"Micro-average (AUC = {micro_curve_auc:.3f})")
    plt.plot([0, 1], [0, 1], "k--", label="Chance")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC curves")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

    return {
        "observed_counts": observed_counts,
        "expected_counts_mean": expected_means,
        "oe_ratio": oe_ratios,
        "auc_posterior": {
            name: summarize(values) for name, values in auc_draws.items()
        },
        "roc_auc_from_mean_probabilities": {
            "macro": float(macro_curve_auc),
            "micro": float(micro_curve_auc),
        },
    }

In [ ]:
import numpy as np
import pandas as pd
import pymc as pm
import matplotlib.pyplot as plt

from scipy.special import expit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import auc, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from pathlib import Path


# Load data and split before preprocessing to avoid test-data leakage.
filename = "healthcare_dataset_2_missingness.csv"
repo_root = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / "data" / filename).is_file()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        f"Could not find data/{filename} from notebook directory {Path.cwd()}"
    )
path = repo_root / "data" / filename
df = pd.read_csv(path)
X = df[["age", "gender", "chief_complaint", "saturation_mcar"]]
y = df["mortality_30_days"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

preprocess = ColumnTransformer(
    [
        (
            "numeric",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                    ("scale", StandardScaler()),
                ]
            ),
            ["age", "saturation_mcar"],
        ),
        (
            "categorical",
            OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
            ["gender", "chief_complaint"],
        ),
    ],
    verbose_feature_names_out=False,
)

X_train = preprocess.fit_transform(X_train)
X_test = preprocess.transform(X_test)

# Bayesian logistic regression:
# mortality ~ Bernoulli(sigmoid(intercept + X @ coefficients))
with pm.Model() as model:
    intercept = pm.Normal("intercept", mu=0, sigma=2.5)
    coefficients = pm.Normal("coefficients", mu=0, sigma=1, shape=X_train.shape[1])

    probability = pm.math.sigmoid(intercept + pm.math.dot(X_train, coefficients))
    pm.Bernoulli("mortality", p=probability, observed=y_train.to_numpy())

    trace = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.9,
        random_seed=42,
    )

# Risk probability for each test patient at each posterior draw.
intercept_draws = trace.posterior["intercept"].values.reshape(-1)
coefficient_draws = trace.posterior["coefficients"].values.reshape(
    -1, X_train.shape[1]
)
risk_draws = expit(
    intercept_draws[:, None] + coefficient_draws @ X_test.T
)

# Observed-to-expected event ratio per posterior draw.
observed_events = int(y_test.sum())
expected_events = risk_draws.sum(axis=1)
oe_draws = observed_events / expected_events

# AUC per posterior draw, plus AUC from mean posterior risk.
auc_draws = np.array([
    roc_auc_score(y_test, risk) for risk in risk_draws
])
mean_risk = risk_draws.mean(axis=0)
auc_mean_risk = roc_auc_score(y_test, mean_risk)

print(f"Test deaths: {observed_events} of {len(y_test)}")
print(
    f"O:E median and 95% interval: "
    f"{np.median(oe_draws):.3f} "
    f"({np.percentile(oe_draws, 2.5):.3f}, "
    f"{np.percentile(oe_draws, 97.5):.3f})"
)
print(
    f"AUC median and 95% interval: "
    f"{np.median(auc_draws):.3f} "
    f"({np.percentile(auc_draws, 2.5):.3f}, "
    f"{np.percentile(auc_draws, 97.5):.3f})"
)
print(f"AUC from mean posterior risk: {auc_mean_risk:.3f}")

# Macro/micro ROC curves, treating the two binary classes one-vs-rest.
y_array = y_test.to_numpy()
y_one_hot = np.column_stack([1 - y_array, y_array])
probabilities = np.column_stack([1 - mean_risk, mean_risk])

class_curves = [
    roc_curve(y_one_hot[:, i], probabilities[:, i])
    for i in range(2)
]
all_fpr = np.unique(np.concatenate([fpr for fpr, _, _ in class_curves]))
macro_tpr = np.mean(
    [np.interp(all_fpr, fpr, tpr) for fpr, tpr, _ in class_curves],
    axis=0,
)
micro_fpr, micro_tpr, _ = roc_curve(y_one_hot.ravel(), probabilities.ravel())

macro_auc = auc(all_fpr, macro_tpr)
micro_auc = auc(micro_fpr, micro_tpr)

plt.figure(figsize=(7, 6))
plt.plot(all_fpr, macro_tpr, label=f"Macro ROC (AUC = {macro_auc:.3f})")
plt.plot(micro_fpr, micro_tpr, label=f"Micro ROC (AUC = {micro_auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Chance")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("30-day mortality ROC")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()